# scratch18b96_pair2_distill_A8

From-scratch **18 blocks x 96 channels** PAIR2 student (+ legal-mask loss, A1 recipe), distilled from teacher **A8**: 12,906,096 states (teacher games, epoch-4 games, crisis states) all labeled with the teacher's 8-symmetry-averaged policy (top-5). HISTORY 258.

Two arms, pick one per runtime in the CONFIG cell: `TARGET = "soft"` (full top-5 distribution) or `TARGET = "hard"` (the teacher's move only).

Drive files (MyDrive/alphatrain/): `colorlines_pillar3d_v8.tar.gz` + `born_again_A8.pt.gz`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, time
DRIVE='/content/drive/MyDrive/alphatrain'
!cp {DRIVE}/colorlines_pillar3d_v8.tar.gz /content/
!cd /content && tar xzf colorlines_pillar3d_v8.tar.gz
os.makedirs('/content/alphatrain/data', exist_ok=True)
t0=time.time()
!cp {DRIVE}/born_again_A8.pt.gz /content/born_again_A8.pt.gz
gz=os.path.getsize('/content/born_again_A8.pt.gz'); print(f'.gz: {gz:,} bytes')
assert gz == 873544987, f'.gz truncated! got {gz}; re-upload born_again_A8.pt.gz'
!gunzip -t /content/born_again_A8.pt.gz && echo '.gz integrity OK'
!gzip -dc /content/born_again_A8.pt.gz > /content/alphatrain/data/born_again_A8.pt
pt=os.path.getsize('/content/alphatrain/data/born_again_A8.pt')
assert pt == 2064979169, f'.pt size wrong! got {pt}'
print(f'corpus: {pt/1e9:.2f} GB, 12,906,096 rows ({time.time()-t0:.0f}s)')
!rm /content/born_again_A8.pt.gz
!pip install -q numpy numba scipy
_src = open('/content/alphatrain/dataset.py').read()
assert 'def _build_line_direction_luts' in _src and 'def _transform_observation' in _src, 'STALE CODE TARBALL (old D4 bug)'
_tp = open('/content/alphatrain/train_path_b.py').read()
assert 'class MixedLoader' in _tp and 'def legal_mask_from_obs' in _tp, 'STALE CODE TARBALL (trainer)'
print('code tarball OK')

In [ ]:
import torch
print(f'PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g=torch.cuda.get_device_properties(0); print(f'GPU {torch.cuda.get_device_name(0)} | {g.total_memory/1e9:.0f} GB')

In [ ]:
# ===== CONFIG =====
TARGET     = "soft"   # "soft": the teacher's top-5 distribution; "hard": the teacher's move only
NUM_BLOCKS = 18
CHANNELS   = 96
EPOCHS     = 40
BATCH      = 32768
LR         = 3e-3
SEED       = 42
SAVE_STEPS = 1000
BLEND      = 1.0 if TARGET == "soft" else 0.0   # train_path_b --blend-alpha: 1 = soft CE, 0 = hard CE
RUN        = f"scratch18b96_pair2_distill_A8_{TARGET}"
print(f'RUN={RUN} ep={EPOCHS} bs={BATCH} lr={LR} blend={BLEND}')

In [ ]:
# Resume from the last COMPLETED epoch if Colab stopped. Checkpoints go straight to Drive every epoch.
LATEST = f'/content/drive/MyDrive/alphatrain/{RUN}_ckpts/latest.pt'
RESUME_ARGS = f'--resume {LATEST}' if os.path.exists(LATEST) else ''
print('Resume:', LATEST if RESUME_ARGS else 'none (new run)')
%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train_path_b \
    {RESUME_ARGS} --tensor-file alphatrain/data/born_again_A8.pt \
    --num-blocks {NUM_BLOCKS} --channels {CHANNELS} --seed {SEED} --amp --compile \
    --epochs {EPOCHS} --batch-size {BATCH} --lr {LR} --warmup-epochs 1 \
    --target-temperature 1.0 --blend-alpha {BLEND} --policy-head pair2 --pair-dim 64 --legal-mask-loss \
    --save-every-steps {SAVE_STEPS} \
    --copy-to /content/drive/MyDrive/alphatrain/{RUN}_best.pt \
    --save-dir /content/drive/MyDrive/alphatrain/{RUN}_ckpts 2>&1 | tee /content/{RUN}_train.log
# val loss is informational only - the gate is the local C++ eval (death rate over 2k games).

In [ ]:
import glob
for f in sorted(glob.glob(f'/content/drive/MyDrive/alphatrain/{RUN}_ckpts/*.pt')):
    print(f)

## Evaluation (local M5, C++ only)

Download epochs as they land (e.g. 7 / 14 / 20 / 27 / 33 / 40) into `alphatrain/data/`, then gate each one
(folded export, 2,000 games, cap 100k, logged to EVAL.md with the death rate):

```bash
python -m alphatrain.scripts.eval_log run --model alphatrain/data/<run>_ckpts_epoch_N.pt --desc "18b96 student of A8, epoch N"
```

Compare the death rate (deaths per 100k turns, 95% CI) with the teacher's; the teacher's 8-view-averaged
policy is the label source, so matching or beating the single-pass teacher is the goal.